# Day 4 Assignments

This notebook contains all four data-analysis assignments in one file.
Run each section individually to view the output.

In [2]:
!pip install pandas
from pathlib import Path
import pandas as pd

base_dir = Path.cwd()
required_files = [
    'patient_clinical_data_raw.csv',
    'ecommerce_orders_raw.csv',
    'iot_sensor_data_raw.csv',
    'customers.csv',
    'products.csv',
    'orders.csv'
]
if not all((base_dir / name).exists() for name in required_files):
    base_dir = base_dir.parent

base_dir

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-2.3.3-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.8 MB)


PosixPath('/home/user01/Assignments/Assignment Day-4')

## Assignment 1: Patient Clinical Data

Clean and analyze the clinical dataset.

In [3]:
df = pd.read_csv(base_dir / 'patient_clinical_data_raw.csv')
print('Shape:', df.shape)
print('\nFirst 10 rows:')
print(df.head(10))
print('\nMissing values:')
print(df.isnull().sum())

num_cols = df.select_dtypes(include='number').columns
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

before = len(df)
df = df.drop_duplicates()
print(f'\nDuplicate rows removed: {before - len(df)}')

df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df.loc[df['Age'].isna() | (df['Age'] < 0), 'Age'] = df['Age'].median()

mapping_gender = {'M': 'Male', 'F': 'Female', 'male': 'Male', 'female': 'Female'}
df['Gender'] = df['Gender'].replace(mapping_gender).str.title().str.strip()
df['Department'] = df['Department'].str.title().str.strip()

bins = [0, 17, 40, 60, float('inf')]
labels = ['Child', 'Young Adult', 'Middle Age', 'Senior']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

bp = df['Blood_Pressure'].str.split('/', expand=True)
bp.columns = ['Systolic', 'Diastolic']
bp = bp.apply(pd.to_numeric, errors='coerce')
df[['Systolic', 'Diastolic']] = bp

df['High_Risk'] = ((df['Heart_Rate'] > 100) | (df['Cholesterol'] > 220) | (df['Glucose'] > 140) | (df['Temperature'] > 99.5) | (df['Age'] > 65)).astype(int)

if 'Treatment_Cost' in df.columns and 'Length_of_Stay' in df.columns:
    df['Cost_Per_Day'] = df['Treatment_Cost'] / df['Length_of_Stay']

print('\nAverage age:', round(df['Age'].mean(), 2))
print('\nPatients per department:')
print(df['Department'].value_counts())
print('\nAverage treatment cost by department:')
print(df.groupby('Department')['Treatment_Cost'].mean().sort_values(ascending=False))
print('\nReadmission percentage:', round(df['Readmission'].str.lower().eq('yes').mean() * 100, 2), '%')

Shape: (10100, 14)

First 10 rows:
  Patient_ID   Age  Gender        Department Blood_Pressure  Heart_Rate  \
0     P07402  58.0  Female       Orthopedics         119/73        64.0   
1     P05835   NaN  Female  General Medicine         107/91        62.0   
2     P02123  47.0    Male       Orthopedics         148/98        83.0   
3     P08789   NaN    Male          Oncology         115/84        89.0   
4     P00305  47.0  Female       Orthopedics         128/62        77.0   
5     P02532  31.0  Female          Oncology         104/85        99.0   
6     P02996  49.0    MALE         Neurology         118/83       108.0   
7     P07660  32.0  Female        Pediatrics         122/89        64.0   
8     P08225  64.0    Male         Neurology        161/100        61.0   
9     P04449  56.0    Male  General Medicine         102/76        83.0   

   Temperature  Cholesterol  Glucose      Diagnosis Admission_Type  \
0        100.0        197.0     60.0  Heart Disease       Referral   